# the reference MS1 ↔ Biomapper — exploration

Interactive view over the latest run's `comparison.csv` + raw JSON + the 4 method sheets.
Run `python run_comparison.py` first. **Clear outputs before committing.**


In [ ]:
import glob, os, json
import pandas as pd
import report as R, compare as C

run = sorted(glob.glob('outputs/2026*'))[-1]
comp = pd.read_csv(os.path.join(run, 'comparison.csv'))
print('run:', run, '| features:', len(comp))
comp.head()


## Concordance summary (name-only, authoritative)


In [ ]:
m = R.aggregate(comp)
rows = []
for ns in R.ALL_NAMESPACES:
    d = m['namespaces'][ns]
    rows.append({'namespace': ns, 'comparable': d['comparable'],
                 'comparable_%': round(100*d['comparable_frac'],1),
                 'exact_%': None if d['exact_rate'] is None else round(100*d['exact_rate'],1),
                 'agreement_%': None if d['agreement_rate'] is None else round(100*d['agreement_rate'],1),
                 'new_coverage': d['new_coverage'], 'missed': d['missed']})
pd.DataFrame(rows)


## Class distribution per namespace


In [ ]:
import io_and_normalize as io
pd.DataFrame({ns: comp[f'{ns}__class'].value_counts() for ns in io.SCORED_NAMESPACES}).fillna(0).astype(int)


## Disagreements — drill down


In [ ]:
ns = 'CHEBI'  # change me
dis = comp[comp[f'{ns}__class'] == C.DISAGREE]
dis[['feature_id','matched_name','match_level',f'{ns}__ref',f'{ns}__bmap','confidence_tier']]


## New coverage (UNVALIDATED) — by confidence tier
No ground truth here; spot-check before trusting.


In [ ]:
ns = 'KEGG.COMPOUND'  # change me
nc = comp[comp[f'{ns}__class'] == C.NEW_COVERAGE]
print(nc['confidence_tier'].value_counts().to_dict())
nc[['feature_id','matched_name','match_level',f'{ns}__bmap','confidence_tier']].head(30)


## Per-method breakdown (from the xlsx)
A feature_id may appear in multiple method sheets.


In [ ]:
xls = pd.ExcelFile('data/All_Methods_Features.xlsx')
method_of = {}
for sheet in xls.sheet_names:
    s = xls.parse(sheet, usecols=['feature_id'])
    for fid in s['feature_id'].astype(str):
        method_of.setdefault(fid, set()).add(sheet)

rows = []
for _, r in comp.iterrows():
    for method in method_of.get(str(r['feature_id']), {'(none)'}):
        rows.append({'method': method, 'CHEBI__class': r['CHEBI__class']})
mdf = pd.DataFrame(rows)
agree = mdf['CHEBI__class'].isin([C.AGREE_EXACT, C.AGREE_PARTIAL])
comparable = mdf['CHEBI__class'].isin([C.AGREE_EXACT, C.AGREE_PARTIAL, C.DISAGREE])
summary = mdf.assign(agree=agree, comparable=comparable).groupby('method')[['agree','comparable']].sum()
summary['agreement_%'] = (100*summary['agree']/summary['comparable']).round(1)
summary
